# **Space X  Falcon 9 First Stage Landing Prediction**
## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia
In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)

Falcon 9 first stage will land successfully
![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)

Several examples of an unsuccessful landing are shown here:
![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)

More specifically, the launch records are stored in a HTML table shown below:
![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)

  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame

In [67]:
import requests
from bs4 import BeautifulSoup
import unicodedata
import pandas as pd

y te proporcionaremos algunas funciones auxiliares para que puedas procesar tablas HTML obtenidas de la web

In [ ]:
def date_time(table_cells):
    # Extrae los valores de fecha y hora de la celda de la tabla
    values = [text.strip() for text in table_cells.strings]
    return values[:2]

def booster_version(table_cells):
    # Extrae la versión del propulsor de la celda de la tabla
    parts = [text for i, text in enumerate(table_cells.strings) if i % 2 == 0]
    return " ".join(parts[:-1])

def landing_status(table_cells):
    # Extrae el estado de aterrizaje de la celda de la tabla
    return next(table_cells.strings).strip()

def get_mass(table_cells):
    # Extrae la masa de la celda de la tabla
    mass_text = unicodedata.normalize("NFKD", table_cells.get_text()).strip()
    kg_pos = mass_text.find("kg")
    return mass_text[:kg_pos + 2] if kg_pos != -1 else 0

def extract_column_from_header(row):
    for tag in (row.br, row.a, row.sup):
        if tag:
            tag.extract()

    column_name = " ".join(row.contents).strip()

    # Ignore numeric-only headers and empty text
    if column_name and not column_name.isdigit():
        return column_name

In [69]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

### TASK 1: Request the Falcon9 Launch Wiki page from its URL
First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.

In [70]:
# use requests.get() method with the provided static_url and headers
page = requests.get(static_url, headers=headers)
# assign the response to a object called soup using BeautifulSoup
soup = BeautifulSoup(page.content, "html.parser")
# Use soup.title attribute
soup.title

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

### TASK 2: Extract all column/variable names from the HTML table header
Next, we want to collect all relevant column names from the HTML table header
Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab

In [71]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
html_tables = soup.find_all("table")

Starting from the third table is our target table contains the actual launch records.

In [72]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table.prettify())

<table class="wikitable plainrowheaders collapsible" style="width: 100%;">
 <tbody>
  <tr>
   <th scope="col">
    Flight No.
   </th>
   <th scope="col">
    Date and
    <br/>
    time (
    <a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">
     UTC
    </a>
    )
   </th>
   <th scope="col">
    <a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">
     Version,
     <br/>
     Booster
    </a>
    <sup class="reference" id="cite_ref-booster_11-0">
     <a href="#cite_note-booster-11">
      <span class="cite-bracket">
       [
      </span>
      b
      <span class="cite-bracket">
       ]
      </span>
     </a>
    </sup>
   </th>
   <th scope="col">
    Launch site
   </th>
   <th scope="col">
    Payload
    <sup class="reference" id="cite_ref-Dragon_12-0">
     <a href="#cite_note-Dragon-12">
      <span class="cite-bracket">
       [
      </span>
      c
      <span class="cite-bracket">
       ]
  

You should able to see the columns names embedded in the table header elements `<th>` as follows:
<br> Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [73]:
# Extraer nombres de columnas validos del encabezado
column_names = [
    name
    for th in first_launch_table.find_all("th")
    for name in [extract_column_from_header(th)]
    if name
]
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables
Crearemos un diccionario vacío con claves a partir de los nombres de columnas extraídos en la tarea anterior. Más adelante, este diccionario se convertirá en un dataframe de Pandas.

In [74]:
# Estructura final esperada para el dataframe
launch_dict = {
    'Flight No.': [],
    'Date': [],
    'Time': [],
    'Version Booster': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Booster landing': []
}

A continuación, solo necesitamos llenar el `launch_dict` con los registros de lanzamientos extraídos de las filas de la tabla. Por lo general, las tablas HTML en las páginas de Wiki pueden contener anotaciones inesperadas y otros tipos de ruidos, como enlaces de referencia `B0004.1[8]`, valores faltantes `N/A [e]`, formatos inconsistentes, etc.

Para simplificar el proceso de análisis, hemos proporcionado un fragmento de código incompleto a continuación para ayudarte a llenar el `launch_dict`. Por favor, completa el siguiente fragmento de código con los TODOs o puedes optar por escribir tu propia lógica para analizar todas las tablas de lanzamientos:

In [75]:
extracted_row = 0
tables = soup.find_all("table", "wikitable plainrowheaders collapsible")

for table in tables:
    for row_html in table.find_all("tr"):
        flight_header = row_html.find("th")
        cells = row_html.find_all("td")

        # Validar que sea una fila de lanzamiento completa
        if not flight_header or not flight_header.text.strip().isdigit() or len(cells) < 9:
            continue

        extracted_row += 1
        flight_no = int(flight_header.text.strip())
        date, time = date_time(cells[0])
        version = booster_version(cells[1]) or cells[1].get_text(strip=True)

        launch_dict['Flight No.'].append(flight_no)
        launch_dict['Date'].append(date.strip(","))
        launch_dict['Time'].append(time)
        launch_dict['Version Booster'].append(version)
        launch_dict['Launch site'].append(cells[2].get_text(strip=True))
        launch_dict['Payload'].append(cells[3].get_text(strip=True))
        launch_dict['Payload mass'].append(get_mass(cells[4]))
        launch_dict['Orbit'].append(cells[5].get_text(strip=True))
        launch_dict['Customer'].append(cells[6].get_text(strip=True))
        launch_dict['Launch outcome'].append(landing_status(cells[7]).strip())
        launch_dict['Booster landing'].append(landing_status(cells[8]).strip())

After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.

In [76]:
df = pd.DataFrame(launch_dict)

In [77]:
df.head()

,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.0 7 B0003.1 8,"CCAFS,SLC-40",Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure
1,2,8 December 2010,15:43,F9 v1.0 7 B0004.1 8,"CCAFS,SLC-40",Dragondemo flight C1(Dragon C101),0,LEO(ISS),NASA(COTS)NRO,Success,Failure
2,3,22 May 2012,07:44,F9 v1.0 7 B0005.1 8,"CCAFS,SLC-40",Dragondemo flight C2+[18](Dragon C102),525 kg,LEO(ISS),NASA(COTS),Success,No attempt
3,4,8 October 2012,00:35,F9 v1.0 7 B0006.1 8,"CCAFS,SLC-40",SpaceX CRS-1[22](Dragon C103),"4,700 kg",LEO(ISS),NASA(CRS),Success,No attempt
4,5,1 March 2013,15:10,F9 v1.0 7 B0007.1 8,"CCAFS,SLC-40",SpaceX CRS-2[22](Dragon C104),"4,877 kg",LEO(ISS),NASA(CRS),Success,No attempt


Ahora podemos exportarlo a un <b>CSV</b> para la siguiente sección, pero para mantener las respuestas consistentes y en caso de que tengas dificultades para completar este laboratorio.

Los siguientes laboratorios utilizarán un conjunto de datos proporcionado para que cada laboratorio sea independiente.

In [78]:
df.to_csv("spacex_web_scraped.csv", index=False)